# Esfandiar Kiani (40311614) - ML Homework 02.5

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo 

## Load data

In [ ]:
FILE_PATH = "D:/M.A/T1/ML/Assignments/HW2_Regression_code  datasets/abalone.csv"
abalone = fetch_ucirepo(id=1) 

In [ ]:
# df = pd.read_csv(FILE_PATH)

In [ ]:
x = abalone.data.features
y = abalone.data.targets

## Meta data

In [ ]:
print(x.head())
print()
print('_______________________________________________')
print()
print(y.head())

In [ ]:
print(x.info())
print()
print('_______________________________________________')
print()
print(y.info())

In [ ]:
print(x.describe())
print()
print('_______________________________________________')
print()
print(y.describe())

## MSE

In [ ]:
def MSE(y_true, y_pred):
    return sum((true - pred) ** 2 for true, pred in zip(y_true, y_pred)) / len(y_true)

## Data cleansing

In [ ]:
x = x.drop(columns=['Sex'])
print(x.info())

## Train/Test split

In [ ]:
ratio = 0.8

In [ ]:
x_train, x_test = x[:int(ratio*len(x))], x[int(ratio*len(x)):]
y_train, y_test = y[:int(ratio*len(y))], y[int(ratio*len(y)):]

In [ ]:
print(x_train.info(), x_test.info())

## Models


### a) Linear

In [ ]:
class LinearRegression:
    def fit(self, X, y):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        self.theta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y

    def predict(self, X):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return X_b @ self.theta

### b&c) Polynomial

In [ ]:
class PolynomialRegression:
    def __init__(self, degree):
        self.degree = degree

    def fit(self, X, y):
        self.X_poly = self._polynomial_features(X)
        # self.theta = np.linalg.inv(self.X_poly.T @ self.X_poly) @ self.X_poly.T @ y
        #! Convert inv to pinv due to singular matrix error
        self.theta = np.linalg.pinv(self.X_poly.T @ self.X_poly) @ self.X_poly.T @ y

    def predict(self, X):
        X_poly = self._polynomial_features(X)
        return X_poly @ self.theta

    def _polynomial_features(self, X):
        return np.hstack([X ** i for i in range(self.degree + 1)])

### d) RBF kernel functions

In [ ]:
class RBFKernelRegression:
    def __init__(self, gamma):
        self.gamma = gamma

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def predict(self, X):
        K = np.exp(-self.gamma * self._squared_distances(X, self.X_train))
        return K @ self.y_train / np.sum(K, axis=1)

    def _squared_distances(self, X1, X2):
        return np.sum(X1**2, axis=1).reshape(-1, 1) + np.sum(X2**2, axis=1) - 2 * X1 @ X2.T

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Polynomial Regression (degree=2)': PolynomialRegression(degree=2),
    'Polynomial Regression (degree=3)': PolynomialRegression(degree=3),
    'RBF Kernel Regression': RBFKernelRegression(gamma=0.1)
}

## Fit, predict, and evaluation

In [ ]:
for name, model in models.items():
    
    model.fit(x_train.values, y_train.values)
    
    y_train_pred = model.predict(x_train.values)
    y_test_pred = model.predict(x_test.values)
    
    train_mse = MSE(y_train.values, y_train_pred)
    test_mse = MSE(y_test.values, y_test_pred)
    
    print(f"{name}: Train MSE = {train_mse}, Test MSE = {test_mse}")